# Auditoría de memoria GPU por arquitectura y batch (sin entrenamiento)\n\nMide DeepLabV3+ semántico y Mask R-CNN de instancias a 512×512, con batch 1, 2 y 4. Cada combinación se ejecuta en un proceso aislado para contener errores OOM. Usa tensores sintéticos: **no descarga ni lee datasets, no toca test, no guarda checkpoints y nunca llama `optimizer.step()`**. Sí realiza un `forward` y un `backward` para medir memoria de activaciones de forma representativa.\n\nEl resultado orienta el batch inicial; no demuestra estabilidad de una época completa ni autoriza entrenamiento. La memoria real puede variar con imágenes, número de objetos, augmentations, workers y validación.

## 1. Montar Google Drive y validar el preflight aprobado\n\nLos informes se guardarán en `MyDrive/ProcesamientoImagenes/auditoria_memoria_gpu`.

In [ ]:
from google.colab import drive\nfrom pathlib import Path\nimport os, sys, json, subprocess, time\nimport pandas as pd\ndrive.mount('/content/drive')\nOUT=Path('/content/drive/MyDrive/ProcesamientoImagenes/auditoria_memoria_gpu')\nOUT.mkdir(parents=True,exist_ok=True)\nPREFLIGHT=Path('/content/drive/MyDrive/ProcesamientoImagenes/auditoria_entorno_entrenamiento/training_runtime_preflight.json')\nif not PREFLIGHT.is_file(): raise RuntimeError('No encuentro training_runtime_preflight.json. Ejecuta primero el preflight de entorno.')\npre=json.loads(PREFLIGHT.read_text())\nif pre.get('status')!='PASS' or pre.get('gpu_status')!='PASS' or pre.get('drive_status')!='PASS': raise RuntimeError('El preflight GPU/Drive no está aprobado; detengo la auditoría.')\nprint('Preflight aprobado:',pre.get('torch'),pre.get('devices',[{}])[0].get('name'))\n

## 2. Sincronizar el repositorio privado\n\nUsa `GITHUB_TOKEN` en Colab Secrets (Contents: Read-only). No pegues el token en el notebook ni lo compartas.

In [ ]:
from getpass import getpass\nREPO='Leohf04/Procesamiento-de-Im-genes-'\nBRANCH='urban-models-and-benchmark'\nWORKDIR=Path('/content/Procesamiento-de-Im-genes-')\ntry:\n from google.colab import userdata\n TOKEN=userdata.get('GITHUB_TOKEN')\nexcept Exception: TOKEN=None\nif not TOKEN: TOKEN=getpass('GitHub PAT (Contents: Read-only): ')\nif not TOKEN: raise RuntimeError('Falta GITHUB_TOKEN')\naskpass=Path('/tmp/colab_git_askpass.sh')\naskpass.write_text('#!/bin/sh\\ncase \\"$1\\" in\\n *Username*) echo x-access-token ;;\\n *Password*) echo \\"$COLAB_GITHUB_TOKEN\\" ;;\\n *) exit 1 ;;\\nesac\\n'); askpass.chmod(0o700)\nenv=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})\nif WORKDIR.exists():\n subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env); subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True); subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)\nelse: subprocess.run(['git','clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)\ndel TOKEN,env\nos.chdir(WORKDIR)\nHEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()\nprint('HEAD:',HEAD)\nif not Path('scripts/audit_model_memory.py').is_file(): raise RuntimeError('Falta scripts/audit_model_memory.py en el HEAD clonado.')\n

## 3. Ejecutar combinaciones aisladas\n\nCada proceso instancia el modelo real de la arquitectura, usa datos sintéticos 512×512, ejecuta forward/backward y termina sin optimizar pesos. Orden: batch 1, 2 y 4 por tarea. Si una combinación da OOM, el proceso termina y la siguiente se intenta en un proceso nuevo. Se mide AMP desactivado para coincidir con la configuración conservadora actual del protocolo.

In [ ]:
CASES=[(task,batch) for task in ('semantic','instances') for batch in (1,2,4)]\nrows=[]\nfor task,batch in CASES:\n out=OUT/f'{task}_b{batch}_512.json'\n cmd=[sys.executable,'scripts/audit_model_memory.py','--task',task,'--batch-size',str(batch),'--image-size','512','--output',str(out)]\n print('\\n'+'='*72+'\\n',task,'batch=',batch,'\\n'+'='*72)\n proc=subprocess.run(cmd,cwd=WORKDIR,text=True,capture_output=True)\n print(proc.stdout)\n if proc.stderr: print('STDERR:',proc.stderr[-3000:])\n if not out.is_file(): raise RuntimeError(f'No se generó el informe para {task}/batch={batch}')\n item=json.loads(out.read_text()); item['process_exit_code']=proc.returncode; rows.append(item)\n if item.get('status')=='FAIL': raise RuntimeError(f'Error no-OOM en {task}/batch={batch}: {item.get(\"error\")}')\n

## 4. Resumen y criterio de selección\n\nNo se debe escoger batch 4 por ser el mayor que cabe en una única pasada. Para el piloto se prioriza batch 1 si es estable; se aumenta solo si deja margen suficiente de VRAM. Mask R-CNN es especialmente sensible al número de propuestas/objetos y estas muestras sintéticas no cubren el peor caso.

In [ ]:
df=pd.DataFrame([{k:r.get(k) for k in ('task','batch_size','image_size','status','gpu','peak_allocated_bytes','peak_reserved_bytes','peak_increment_over_model_bytes','elapsed_seconds','backward_calls','optimizer_steps')} for r in rows])\nfor col in ('peak_allocated_bytes','peak_reserved_bytes','peak_increment_over_model_bytes'):\n df[col+'_GiB']=df[col]/(1024**3)\ndisplay(df)\nsummary={'head':HEAD,'preflight_timestamp_utc':pre.get('timestamp_utc'),'cases':rows,'training_started':False,'optimizer_steps':0,'datasets_loaded':False,'test_split_touched':False}\nsummary_path=OUT/'gpu_memory_audit_summary.json'; summary_path.write_text(json.dumps(summary,indent=2,ensure_ascii=False))\nprint('Informe:',summary_path)\nprint('TRAINING_STARTED: False')\nprint('DATASETS_LOADED: False')\nprint('TEST_SPLIT_TOUCHED: False')\nprint('OPTIMIZER_STEPS: 0')\n